In [142]:
import torch
from torch import nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [143]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")

In [144]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [145]:
df.drop(columns=['id', 'Unnamed: 32'], inplace = True)

In [146]:
print(df.shape)
df.head()

(569, 31)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [147]:
xtrain, xtest, ytrain, ytest = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

##### Scaling

In [148]:
scalar = StandardScaler()
xtrain = scalar.fit_transform(xtrain)
xtest = scalar.transform(xtest)

##### Label Encoding

In [149]:
encoder = LabelEncoder()
ytrain = encoder.fit_transform(ytrain)
ytest = encoder.transform(ytest)

#### Numpy Array to Pytorch Tensors

In [150]:
xtrain = torch.from_numpy(xtrain)
xtest = torch.from_numpy(xtest)
ytrain = torch.from_numpy(ytrain)
ytest = torch.from_numpy(ytest)

xtrain = xtrain.float()
xtest = xtest.float()
ytrain = ytrain.float()
ytest = ytest.float()

In [151]:
print(f"X_train shape: {xtrain.shape}")
print(f"X_test shape: {xtest.shape}")
print(f"Y_train shape: {ytrain.shape}")
print(f"Y_test shape: {ytest.shape}")

X_train shape: torch.Size([455, 30])
X_test shape: torch.Size([114, 30])
Y_train shape: torch.Size([455])
Y_test shape: torch.Size([114])


#### Defining the Neural Netwwork

In [152]:
class MySimpleNeuralNetwork(nn.Module):
    
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out

#### Important Parameters

In [153]:
learning_rate = 0.1
epochs = 25

#### Define Loss Function

In [154]:
loss_func = nn.BCELoss()

#### Training Pipeline

In [155]:
# create model
model = MySimpleNeuralNetwork(num_features=xtrain.shape[1])

#define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):

    y_pred = model(xtrain) # forward pass

    loss = loss_func(y_pred, ytrain.view(-1, 1)) # compute loss

    #clear gradients
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    #parameter update
    optimizer.step()

    # reset gradients to zero
    optimizer.zero_grad()

    # print loss in each epoch
    print(f"Epoch = {epoch + 1}, Loss = {loss}")

Epoch = 1, Loss = 0.8422539234161377
Epoch = 2, Loss = 0.6373827457427979
Epoch = 3, Loss = 0.5252380967140198
Epoch = 4, Loss = 0.45499613881111145
Epoch = 5, Loss = 0.406383752822876
Epoch = 6, Loss = 0.3704317510128021
Epoch = 7, Loss = 0.3425672650337219
Epoch = 8, Loss = 0.32020553946495056
Epoch = 9, Loss = 0.3017719089984894
Epoch = 10, Loss = 0.28625011444091797
Epoch = 11, Loss = 0.27295395731925964
Epoch = 12, Loss = 0.26140257716178894
Epoch = 13, Loss = 0.2512483596801758
Epoch = 14, Loss = 0.2422334998846054
Epoch = 15, Loss = 0.23416227102279663
Epoch = 16, Loss = 0.22688332200050354
Epoch = 17, Loss = 0.22027753293514252
Epoch = 18, Loss = 0.21424967050552368
Epoch = 19, Loss = 0.20872248709201813
Epoch = 20, Loss = 0.20363281667232513
Epoch = 21, Loss = 0.19892798364162445
Epoch = 22, Loss = 0.19456404447555542
Epoch = 23, Loss = 0.19050373136997223
Epoch = 24, Loss = 0.18671518564224243
Epoch = 25, Loss = 0.1831710934638977


In [156]:
print(model.linear.weight)
print("\n")
print(model.linear.bias)

Parameter containing:
tensor([[ 0.3272,  0.1339,  0.1274,  0.2152,  0.1334,  0.0668,  0.3534,  0.0777,
          0.0764,  0.0466,  0.2243, -0.1387,  0.3423,  0.0470,  0.0456, -0.1380,
          0.1176, -0.0893,  0.1147, -0.0934,  0.4724,  0.1500,  0.1481,  0.4109,
          0.1309,  0.1762,  0.1891,  0.2615,  0.0898,  0.0236]],
       requires_grad=True)


Parameter containing:
tensor([-0.1019], requires_grad=True)


#### Model Evaluation

In [157]:
with torch.no_grad():
    y_pred = model.forward(xtest)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == ytest).float().mean()

print(f"Accuracy = {accuracy.item()}")

Accuracy = 0.5392428636550903
